# A3-M2 · PQ SQ
교안의 같은 번호 단원과 함께 봅니다.

## 07-1. 준비 — 2만 개 데이터

In [1]:
import faiss, time
import numpy as np

rng = np.random.default_rng(0)
d, n, k = 64, 20000, 10
centers = rng.normal(0, 10, (50, d)).astype("float32")          # 숨은 주제 50개
xb = (centers[rng.integers(0, 50, n)] + rng.normal(0, 1, (n, d))).astype("float32")   # 문서 2만 개
xq = (centers[rng.integers(0, 50, 100)] + rng.normal(0, 1, (100, d))).astype("float32")  # 질문 100개

flat = faiss.IndexFlatL2(d)
flat.add(xb)
t = time.perf_counter(); _, I_true = flat.search(xq, k); t_flat = time.perf_counter() - t

def recall(I):                     # 정답(전수 비교) k개 중 몇 개를 찾았나, 평균
    return np.mean([len(set(I[i]) & set(I_true[i])) / k for i in range(len(I))])

print(f"문서 {xb.shape}, 질문 {xq.shape}, Flat 검색 {t_flat*1000:.1f} ms (질문 100개)")

문서 (20000, 64), 질문 (100, 64), Flat 검색 19.6 ms (질문 100개)


## 07-2. PQ를 작게 — 4차원 벡터를 2조각으로

In [2]:
rng2 = np.random.default_rng(1)
xt = rng2.normal(0, 1, (2000, 4)).astype("float32")   # 학습용 4차원 벡터
pq = faiss.ProductQuantizer(4, 2, 2)                  # d=4, M=2조각, nbits=2 → 조각마다 대표값 2²=4개
pq.train(xt)
print("조각 수 M:", pq.M, "| 조각 차원 dsub:", pq.dsub, "| 대표값 수 ksub:", pq.ksub)

codebook = faiss.vector_to_array(pq.centroids).reshape(pq.M, pq.ksub, pq.dsub)
for m in range(pq.M):
    print(f"조각 {m} 코드북 (대표값 {pq.ksub}개):", codebook[m].astype(float).round(2).tolist())

v = xt[0]
print("\n원본 벡터:", v.astype(float).round(2).tolist())
code = []
for m in range(pq.M):
    sub = v[m * pq.dsub:(m + 1) * pq.dsub]                     # m번째 조각
    dist = ((codebook[m] - sub) ** 2).sum(axis=1)              # 대표값 4개까지 거리
    code.append(int(dist.argmin()))
    print(f"조각 {m} {sub.astype(float).round(2).tolist()} → 대표값 거리 {dist.astype(float).round(2).tolist()} → 번호 {code[-1]}")
print("손으로 만든 코드:", code)

approx = np.concatenate([codebook[m][code[m]] for m in range(pq.M)])
print("코드로 되살린 벡터:", approx.astype(float).round(2).tolist())
print("faiss decode     :", pq.decode(pq.compute_codes(v[None]))[0].astype(float).round(2).tolist())

조각 수 M: 2 | 조각 차원 dsub: 2 | 대표값 수 ksub: 4
조각 0 코드북 (대표값 4개): [[1.11, -0.11], [0.02, 1.1], [-0.15, -1.21], [-1.02, 0.05]]
조각 1 코드북 (대표값 4개): [[0.74, -1.06], [0.77, 0.61], [-0.8, -0.59], [-0.8, 1.03]]

원본 벡터: [0.35, 0.82, 0.33, -1.3]
조각 0 [0.35, 0.82] → 대표값 거리 [1.45, 0.18, 4.37, 2.47] → 번호 1
조각 1 [0.33, -1.3] → 대표값 거리 [0.23, 3.85, 1.8, 6.72] → 번호 0
손으로 만든 코드: [1, 0]
코드로 되살린 벡터: [0.02, 1.1, 0.74, -1.06]
faiss decode     : [0.02, 1.1, 0.74, -1.06]


## 07-3. 압축된 벡터와 거리 재기 — 거리표(ADC)

In [3]:
codes_db = [[int(((codebook[m] - xt[i, m*2:(m+1)*2]) ** 2).sum(axis=1).argmin()) for m in range(2)]
            for i in range(5)]                                # DB 벡터 5개의 코드
qv = rng2.normal(0, 1, 4).astype("float32")
table = np.array([((codebook[m] - qv[m*2:(m+1)*2]) ** 2).sum(axis=1) for m in range(2)])  # [M, ksub]
print("질문 기준 거리표 (행=조각, 열=대표값 번호):\n", table.round(2))
for i, c in enumerate(codes_db):
    approx = table[0, c[0]] + table[1, c[1]]               # 표에서 두 칸만 꺼내 더한다
    exact = float(((xt[i] - qv) ** 2).sum())
    print(f"DB {i}번 코드 {c} → 근사 거리 {approx:.2f} / 실제 거리 {exact:.2f}")

질문 기준 거리표 (행=조각, 열=대표값 번호):
 [[ 1.7   8.23  2.8   8.42]
 [ 2.78 11.06  7.62 17.1 ]]
DB 0번 코드 [1, 0] → 근사 거리 11.01 / 실제 거리 8.45
DB 1번 코드 [0, 3] → 근사 거리 18.79 / 실제 거리 16.67
DB 2번 코드 [0, 1] → 근사 거리 12.76 / 실제 거리 15.50
DB 3번 코드 [3, 3] → 근사 거리 25.52 / 실제 거리 19.53
DB 4번 코드 [2, 2] → 근사 거리 10.42 / 실제 거리 12.36


## 07-4. SQ(스칼라 양자화) — 숫자 하나를 1바이트로

In [4]:
sq = faiss.ScalarQuantizer(4, faiss.ScalarQuantizer.QT_8bit)
sq.train(xt)
tr = faiss.vector_to_array(sq.trained)                  # 차원별 [최소값 4개, 범위 4개]
vmin, vdiff = tr[:4], tr[4:]
print("차원별 최소값:", vmin.astype(float).round(2).tolist())
print("차원별 범위  :", vdiff.astype(float).round(2).tolist())

c = sq.compute_codes(v[None])[0]                         # 4차원 → 4바이트
print("원본  :", v.astype(float).round(3).tolist())
print("코드  :", c.tolist(), "(0~255)")
print("손계산:", np.round((v - vmin) / vdiff * 255).astype(int).tolist())
print("복원  :", sq.decode(c[None])[0].astype(float).round(3).tolist())

차원별 최소값: [-3.08, -3.25, -3.55, -3.17]
차원별 범위  : [7.01, 6.43, 6.96, 6.47]
원본  : [0.346, 0.822, 0.33, -1.303]
코드  : [124, 161, 142, 73] (0~255)
손계산: [125, 161, 142, 74]
복원  : [0.343, 0.822, 0.339, -1.306]


## 07-5. 2만 개 실험 — IVFPQ·IVF+SQ 크기와 정확도

In [5]:
nlist = 100
for m in [8, 16, 32]:                                  # d=64를 m조각으로
    pq_ix = faiss.IndexIVFPQ(faiss.IndexFlatL2(d), d, nlist, m, 8)
    pq_ix.train(xb); pq_ix.add(xb); pq_ix.nprobe = 8
    _, I = pq_ix.search(xq, k)
    print(f"IVFPQ m={m:<2} 벡터당 {pq_ix.code_size:>2}바이트 recall={recall(I):.3f}")

sq_ix = faiss.IndexIVFScalarQuantizer(faiss.IndexFlatL2(d), d, nlist, faiss.ScalarQuantizer.QT_8bit)
sq_ix.train(xb); sq_ix.add(xb); sq_ix.nprobe = 8
_, I = sq_ix.search(xq, k)
print(f"IVF+SQ8     벡터당 {sq_ix.code_size:>2}바이트 recall={recall(I):.3f}  (원본은 {d*4}바이트)")

IVFPQ m=8  벡터당  8바이트 recall=0.379
IVFPQ m=16 벡터당 16바이트 recall=0.643
IVFPQ m=32 벡터당 32바이트 recall=0.865
IVF+SQ8     벡터당 64바이트 recall=0.979  (원본은 256바이트)
